# E-Commerce Sales Forecasting

This project uses 36 months of historical sales data from an electronics company to forecast the next 6 months. The work includes monthly data preparation, outlier handling, stationarity testing, trend and seasonality decomposition, SARIMA and XGBoost model comparison using RMSE, and a final 6-month forecast with 95% prediction intervals.

## 1. Load and Prepare Monthly Sales Data

The original CSV contains individual sales transactions. We convert the transaction-level data into monthly total sales because the forecasting task requires a monthly time series.

In [ ]:
import os
import json
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

DATA_FILE = "data/ecommerce_sales_data.csv"
OUTPUT_DIR = "outputs"
MODEL_DIR = "models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(DATA_FILE)

df["Order Date"] = pd.to_datetime(df["Order Date"])

monthly_sales = (
    df.set_index("Order Date")["Sales"]
    .resample("MS")
    .sum()
)

print("Number of months:", len(monthly_sales))
print("Start date:", monthly_sales.index.min().date())
print("End date:", monthly_sales.index.max().date())

## 2. Handle Monthly Outliers

We use the IQR rule on the monthly sales totals. Unusual monthly values are clipped to the calculated limits so that extreme observations do not have too much influence on the models.

In [ ]:
q1 = monthly_sales.quantile(0.25)
q3 = monthly_sales.quantile(0.75)
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

outlier_mask = (
    (monthly_sales < lower_limit)
    | (monthly_sales > upper_limit)
)

outliers_handled = int(outlier_mask.sum())

monthly_sales_clean = monthly_sales.clip(
    lower=lower_limit,
    upper=upper_limit
)

print("Monthly outliers detected:", outliers_handled)

## 3. Check Stationarity and Decompose the Series

The Augmented Dickey-Fuller test checks whether the monthly sales series is stationary. We also decompose the series to examine its trend, seasonal pattern, and residual component.

In [ ]:
adf_result = adfuller(monthly_sales_clean)

adf_statistic = adf_result[0]
adf_pvalue = adf_result[1]

print("ADF Statistic:", adf_statistic)
print("ADF p-value:", adf_pvalue)

if adf_pvalue < 0.05:
    print("The monthly sales series is stationary.")
else:
    print("The monthly sales series is not stationary.")

decomposition = seasonal_decompose(
    monthly_sales_clean,
    model="additive",
    period=12
)

decomposition.plot()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "decomposition.png"
    ),
    dpi=150
)

plt.show()
plt.close()

## 4. Split the Data for Model Validation

The first 30 months are used for training and the final 6 known months are used for validation. This allows the forecasting models to be compared before forecasting the future 6 months.

In [ ]:
train_data = monthly_sales_clean.iloc[:30]
validation_data = monthly_sales_clean.iloc[30:]

print("Training months:", len(train_data))
print("Validation months:", len(validation_data))

## 5. Tune SARIMA Models

Several SARIMA `(p,d,q)` combinations and seasonal configurations with a 12-month period are tested. RMSE on the validation period is used to select the best SARIMA configuration.

In [ ]:
sarima_results = []

orders = list(
    itertools.product(
        [0, 1, 2],
        [0, 1],
        [0, 1, 2]
    )
)

seasonal_orders = [
    (0, 0, 0, 12),
    (1, 0, 0, 12),
    (0, 0, 1, 12),
    (1, 0, 1, 12),
    (0, 1, 0, 12),
    (1, 1, 0, 12),
    (0, 1, 1, 12),
    (1, 1, 1, 12)
]

for order in orders:
    for seasonal_order in seasonal_orders:

        try:
            model = SARIMAX(
                train_data,
                order=order,
                seasonal_order=seasonal_order,
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted_model = model.fit(disp=False)

            forecast = fitted_model.forecast(
                steps=len(validation_data)
            )

            rmse = np.sqrt(
                mean_squared_error(
                    validation_data,
                    forecast
                )
            )

            sarima_results.append({
                "order": order,
                "seasonal_order": seasonal_order,
                "rmse": float(rmse),
                "aic": float(fitted_model.aic)
            })

        except Exception:
            continue

sarima_results = sorted(
    sarima_results,
    key=lambda x: x["rmse"]
)

best_sarima = sarima_results[0]

sarima_rmse = best_sarima["rmse"]

print("Best SARIMA model:")
print("Order:", best_sarima["order"])
print("Seasonal order:", best_sarima["seasonal_order"])
print("Validation RMSE:", sarima_rmse)